# The Poincaré conjecture and Ricci flow

*maths · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/maths/poincare_ricci_flow)

Proven: every closed, simply connected 3-manifold is the 3-sphere (Perelman 2002–03, completing Hamilton's Ricci flow programme). Ricci flow $\partial_t g = -2\,\mathrm{Ric}$ smooths a metric like heat smooths temperature. Here it rounds out a batch of lumpy 2-spheres (Gauss–Bonnet $\int K\,dA = 4\pi$ holds to round-off), and pinches the neck of a dumbbell 3-sphere in finite time: the singularity that Perelman's surgery cuts out, after which each piece becomes round.

**Vector form**

$$\frac{\partial g}{\partial t} = -2\,\mathrm{Ric}(g),\qquad \int_{S^2} K\,dA = 4\pi,\qquad g = ds^2 + \psi(s)^2 g_{S^2}:\ \ \partial_t\psi = \psi_{ss} - \frac{1 - \psi_s^2}{\psi}$$

**Index (tensor) form**

$$\partial_t u_{bi} = \bar k_b - K_{bi},\quad K_{bi} = e^{-2u_{bi}}\big(1 - L_{ij}u_{bj}\big),\quad \bar k_b = \frac{4\pi}{2\pi\,w_i e^{2u_{bi}}},\qquad \partial_t u = -(K_\mathrm{rad} + K_\mathrm{tan}) + (u_x + \cot x)\,\xi$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $u_{bi}$ | `u` | (B, N) | conformal factor of each lumpy sphere $b$, metric $e^{2u}(dx^2 + \sin^2x\,d\theta^2)$, at cell $i$ |
| $L_{ij}u_{bj}$ | `einsum('ij,bj->bi', lap, u)` | (B, N) | repeated $j$ ⇒ summed: the flux-form Laplacian of the round sphere; its fluxes vanish at the poles |
| $K_{bi}$ | `gauss_curvature(u, lap)` | (B, N) | Gauss curvature of the conformal metric; Ricci flow on a surface is $\partial_t g = -2Kg$ |
| $2\pi\,w_iK_{bi}e^{2u_{bi}}$ | `2 * pi * einsum('bi,bi,i->b', k, exp(2 * u), w)` | (B,) | total curvature: exactly $4\pi$ for every metric (discrete Gauss–Bonnet), a check at every step |
| $\bar k_b$ | `4 * pi / area(u, w)` | (B,) | mean curvature target of the normalised flow; it keeps each area fixed |
| $K_\mathrm{rad},\ K_\mathrm{tan}$ | `s3_curvatures(u, grid)` | (1, N) | sectional curvatures of the 3-sphere $g = e^{2u}(dx^2 + \sin^2x\,g_{S^2})$, $-\psi_{ss}/\psi$ and $(1-\psi_s^2)/\psi^2$ with $\psi = e^u\sin x$ |
| $\xi$ | `sin(x) * (cumsum(...) - c)` | (1, N) | a diffeomorphism (gauge) term that keeps the metric in conformal form; it changes coordinates, not the geometry |
| $\psi_\mathrm{neck}$ | `neck_radius(psi)` | (1,) | radius of the smallest $S^2$ cross-section; $\psi_\mathrm{neck}^2$ falls at about the cylinder rate $2$ until the pinch |
| $\text{surgery}$ | `surgery(u, grid)` | (1, N) | cut off the high-curvature neck and glue in a round cap tangent to the profile (a cartoon of Perelman's surgery) |

Status: proven. Poincaré asked in 1904 whether a closed 3-manifold in which every loop can be shrunk to a point must be the 3-sphere. Hamilton introduced Ricci flow in 1982 and proved that a 3-manifold with positive Ricci curvature flows to a round metric; in two dimensions every metric on the sphere becomes round (Hamilton 1988, Chow 1991), which the lumpy spheres show: the spread of $K$ decays to zero while $\int K\,dA = 4\pi$ and the area stay fixed. The $l = 1$ harmonics are conformal motions of the round sphere and do not decay, but they do not change the shape. In three dimensions the flow can develop singularities. For a rotationally symmetric dumbbell with a thin enough neck, Angenent and Knopf (2004) proved a neckpinch: the neck shrinks like a cylinder $S^2 \times \mathbb R$, $\psi_\mathrm{neck}^2 \approx 2(T - t)$, while the bulbs stay large. Perelman's papers (arXiv 2002–03) controlled all singularities in three dimensions (no local collapsing, canonical neighbourhoods), defined Ricci flow with surgery that cuts necks and caps them, and showed that for a simply connected manifold the flow goes extinct in finite time, every piece being a round sphere just before it vanishes, so the manifold was a 3-sphere. The proof was checked in detail by Kleiner–Lott, Morgan–Tian and Cao–Zhu; Perelman declined the 2006 Fields Medal and the 2010 Clay Millennium Prize. Numerics here: the surfaces use a finite-volume grid in the polar angle, so discrete Gauss–Bonnet is exact. The 3-sphere is written in conformal form, which is regular at the poles (the arclength form divides by $\psi \to 0$ there and is unstable), with cells packed towards the poles because the bulbs are squeezed there as the neck thins; the flow stops at a neck radius of $0.05$, where the bulbs are still resolved. The surgery here is only a cartoon of Perelman's construction. Try a fat neck ($\varepsilon \gtrsim 0.5$): it does not pinch, and the whole sphere rounds out.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/maths/poincare_ricci_flow/main.py
"""Main loop: parameters → tensors → the two Ricci flows (and surgery) → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import curvature_spread, profile, s3_curvatures, total_curvature

SURGERY_SCALE = 0.05  # neck radius at which we stop and cut
END_RADIUS = 0.35  # stop the capped piece when its largest S² has this radius (it is about to vanish)


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n = 48 if fast_mode() else params["cells"] // 2 * 2
    n_frames = 8 if fast_mode() else 60

    # 1. lumpy 2-spheres under normalised Ricci flow
    x, w, lap = tensors.sphere_grid(n, dev)  # (N,), (N,), (N, N)
    u0 = tensors.lumpy_factors(x)  # (B, N)
    surf = solve.flow_surface(u0, w, lap, 0.3 if fast_mode() else params["t_surface"], n_save=n_frames)
    gb = torch.stack([total_curvature(k, u, w) for k, u in zip(surf["K"], surf["u"])])  # (S, B)
    spread = torch.stack([curvature_spread(k) for k in surf["K"]])  # (S, B)
    print(f"surfaces: {u0.shape[0]} lumpy spheres on {n} cells, normalised Ricci flow to t = {surf['t'][-1]:.2f}")
    print(
        f"  Gauss–Bonnet ∫K dA / 4π over the whole run: {(gb / (4 * math.pi)).min():.13f} … "
        f"{(gb / (4 * math.pi)).max():.13f}"
    )
    print(
        f"  curvature spread (max K − min K)/mean K: {', '.join(f'{v:.2f}' for v in spread[0].tolist())} → "
        f"{', '.join(f'{v:.1e}' for v in spread[-1].tolist())};  area drift "
        f"{((surf['area'][-1] / surf['area'][0]) - 1).abs().max():.1e}"
    )

    # 2. a dumbbell 3-sphere: neckpinch, surgery, and the capped piece rounding out
    grid = tensors.s3_grid(n, dev)  # (x, x_η, x_ηη, h)
    t_max = 0.01 if fast_mode() else 1.0
    pre = solve.flow_s3(tensors.dumbbell(params["neck"], grid[0]), grid, t_max, SURGERY_SCALE, 25, END_RADIUS)
    pinched = pre["neck"][-1].item() < SURGERY_SCALE
    print(f"3-sphere: dumbbell with neck radius ε = {params['neck']}, {len(pre['t'])} Euler steps")
    psi0, psi1 = profile(pre["u"][0], grid)[1], profile(pre["u"][-1], grid)[1]
    post = None
    if pinched:
        t_pinch = pre["t"][-1].item()
        neck = pre["neck"]
        i = int(torch.searchsorted(-neck, torch.tensor(-0.1, dtype=neck.dtype)))
        slope = ((neck[-2] ** 2 - neck[i] ** 2) / (pre["t"][-2] - pre["t"][i])).item()
        print(
            f"  neckpinch: neck radius {neck.max():.3f} → {neck[-1]:.3f} at t = {t_pinch:.4f}, while the largest "
            f"S² only went {psi0.max():.2f} → {psi1.max():.2f}"
        )
        print(f"  d(ψ_neck²)/dt over the last stretch = {slope:.2f}  (a shrinking cylinder S² × R: −2)")
        capped = solve.surgery(pre["u"][-1], grid)
        post = solve.flow_s3(capped, grid, 0.002 if fast_mode() else 1.0, 0.0, 400, END_RADIUS)
        post["t"] += t_pinch
        post["t_save"] += t_pinch
        ratio = []
        for u in post["u"]:
            k = torch.cat(s3_curvatures(u, grid), dim=1)
            ratio.append((k.max() / k.min()).item() if k.min() > 0 else float("nan"))
        post["ratio"] = torch.tensor(ratio, dtype=torch.float64)
        print(f"  surgery at ψ = {SURGERY_SCALE}: cut the neck, glue round caps; each piece then flows on its own")
        print(
            f"  capped piece: K_max/K_min = {post['ratio'][~post['ratio'].isnan()][0]:.2f} → {post['ratio'][-1]:.2f}"
            f" by t = {post['t'][-1]:.3f} (radius {END_RADIUS}): it becomes round as it shrinks away"
        )
    elif psi1.max().item() < END_RADIUS:
        print(
            f"  no neckpinch: the neck opened up and the whole sphere rounded out (radius {END_RADIUS} at "
            f"t = {pre['t'][-1]:.3f})"
        )
    else:
        print(f"  stopped at t = {pre['t'][-1]:.3f} before any pinch (neck radius {pre['neck'][-1]:.3f})")
    return plot.animate(x, surf, spread, gb, grid, pre, post, n_frames, out_dir / "poincare_ricci_flow.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/maths/poincare_ricci_flow/equation.py
r"""Mathematics → tensor translation: Ricci flow ∂g/∂t = −2 Ric in two symmetric settings.

1. Surfaces: on a 2-sphere g = e^{2u} g_round, Ric = K g, and the normalised flow ∂g/∂t = 2(k̄ − K) g (k̄ = 4π/A
   keeps the area A) is the scalar PDE

       ∂u/∂t = k̄ − K,      K = e^{−2u} (1 − Δ_{S²} u),      (Δu)_i = L_ij u_j   (j summed)

   with L the flux-form Laplacian, so ∫K dA = 2π Σ_i (1 − L_ij u_j) w_i = 4π exactly (discrete Gauss–Bonnet).

2. Rotationally symmetric 3-spheres g = ds² + ψ(s)² g_{S²}. In arclength form −2 Ric gives
   ∂ψ/∂t = ψ_ss − (1 − ψ_s²)/ψ (Angenent–Knopf), with sectional curvatures K_rad = −ψ_ss/ψ (planes through ∂_s) and
   K_tan = (1 − ψ_s²)/ψ² (planes tangent to the S² slices). Every such metric is conformally round,
   g = e^{2u(x)}(dx² + sin²x g_{S²}) with ψ = e^u sin x, which is regular at the poles; we evolve u (see s3_rhs).
   The round sphere has K_rad = K_tan = 1/r²; a neck behaves like a cylinder S² × R with K_rad ≈ 0.
"""

import math

import torch


def gauss_curvature(u: torch.Tensor, lap: torch.Tensor) -> torch.Tensor:
    """K_bi = e^{−2u_bi}(1 − L_ij u_bj).   u (B, N), lap (N, N) → (B, N)"""
    return torch.exp(-2 * u) * (1 - torch.einsum("ij,bj->bi", lap, u))


def area(u: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    """A_b = 2π Σ_i e^{2u_bi} w_i.   (B, N) → (B,)"""
    return 2 * math.pi * torch.einsum("bi,i->b", torch.exp(2 * u), w)


def total_curvature(k: torch.Tensor, u: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    """∫K dA = 2π Σ_i K_bi e^{2u_bi} w_i (= 4π for every metric on the sphere).   → (B,)"""
    return 2 * math.pi * torch.einsum("bi,bi,i->b", k, torch.exp(2 * u), w)


def surface_rhs(u: torch.Tensor, w: torch.Tensor, lap: torch.Tensor) -> torch.Tensor:
    """Normalised Ricci flow on S²: ∂u/∂t = k̄ − K with k̄ = 4π / A.   (B, N) → (B, N)"""
    return (4 * math.pi / area(u, w))[:, None] - gauss_curvature(u, lap)


def curvature_spread(k: torch.Tensor) -> torch.Tensor:
    """(max K − min K) / mean K: 0 exactly when the curvature is uniform (a round sphere).   (B, N) → (B,)"""
    return (k.max(dim=-1).values - k.min(dim=-1).values) / k.mean(dim=-1)


def embed_profile(u: torch.Tensor, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Surface of revolution with the metric e^{2u}(dx² + sin²x dθ²): radius ρ = e^u sin x and height
    z = ∫ √(e^{2u} − ρ_x²) dx. Returns ρ, z (B, N) and the slack e^{2u} − ρ_x² (≥ 0 ⇔ embeddable in R³)."""
    rho = torch.exp(u) * torch.sin(x)
    rho_x = torch.gradient(rho, spacing=(x,), dim=-1)[0]
    slack = torch.exp(2 * u) - rho_x**2
    dz = torch.sqrt(slack.clamp_min(0))
    z = torch.cat([torch.zeros_like(dz[:, :1]), torch.cumulative_trapezoid(dz, x, dim=-1)], dim=-1)
    return rho, z - z[:, -1:] / 2, slack


def s3_curvatures(u: torch.Tensor, grid: tuple) -> tuple[torch.Tensor, torch.Tensor]:
    """Sectional curvatures of g = e^{2u}(dx² + sin²x g_{S²}) at the cell centres (u even about both poles):
    K_rad = e^{−2u}(1 − u_xx − u_x cot x),  K_tan = e^{−2u}(1 − u_x² − 2u_x cot x).
    grid = (x, x_η, x_ηη, h): cells uniform in η, x = X(η) denser near the poles (chain rule for ∂_x)."""
    x, xe, xee, h = grid
    ext = torch.cat([u[:, :1], u, u[:, -1:]], dim=1)  # mirror ghost cells: u is even about each pole
    u_e = (ext[:, 2:] - ext[:, :-2]) / (2 * h)
    u_ee = (ext[:, 2:] - 2 * u + ext[:, :-2]) / h**2
    u_x = u_e / xe
    u_xx = (u_ee - xee * u_x) / xe**2
    cot = torch.cos(x) / torch.sin(x)
    e = torch.exp(-2 * u)
    return e * (1 - u_xx - u_x * cot), e * (1 - u_x**2 - 2 * u_x * cot)


def s3_rhs(u: torch.Tensor, grid: tuple) -> torch.Tensor:
    """Ricci flow of g = e^{2u} g_round(S³) kept in this conformal form by a diffeomorphism X = ξ ∂_x (allowed: Ricci
    flow is only defined up to diffeomorphisms):

        ∂u/∂t = −(K_rad + K_tan) + (u_x + cot x) ξ,      ξ = sin x (∫₀^x (K_rad − K_tan)/sin x′ dx′ − c)

    c (half the full integral) keeps the gauge symmetric between the poles.   u (B, N) → (B, N)"""
    x, xe, xee, h = grid
    k_rad, k_tan = s3_curvatures(u, grid)
    integrand = (k_rad - k_tan) / torch.sin(x) * xe * h  # (B, N), vanishes like x² · (1/x) at the poles
    integral = torch.cumsum(integrand, dim=1) - 0.5 * integrand  # midpoint rule up to each centre
    xi = torch.sin(x) * (integral - 0.5 * integrand.sum(dim=1, keepdim=True))
    ext = torch.cat([u[:, :1], u, u[:, -1:]], dim=1)
    u_x = (ext[:, 2:] - ext[:, :-2]) / (2 * h * xe)
    return -(k_rad + k_tan) + (u_x + torch.cos(x) / torch.sin(x)) * xi


def profile(u: torch.Tensor, grid: tuple) -> tuple[torch.Tensor, torch.Tensor]:
    """Arclength s = ∫ e^u dx from the north pole and radius ψ = e^u sin x of the S² slices.   → (B, N) each"""
    x, xe, xee, h = grid
    ds = torch.exp(u) * xe * h
    return torch.cumsum(ds, dim=1) - 0.5 * ds, torch.exp(u) * torch.sin(x)


def neck_radius(psi: torch.Tensor) -> torch.Tensor:
    """Smallest interior local minimum of ψ (the neck); +∞ if ψ has none (a convex, sphere-like profile).   → (B,)"""
    mid = psi[:, 1:-1]
    is_min = (mid < psi[:, :-2]) & (mid < psi[:, 2:])
    return torch.where(is_min, mid, torch.inf).min(dim=-1).values


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/maths/poincare_ricci_flow/tensors.py
"""Build the concrete tensors: the finite-volume grid on the 2-sphere, lumpy conformal factors, and profile metrics
on the 3-sphere (round and dumbbell)."""

import math

import torch


def sphere_grid(n: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Polar angle x_j = (j + ½)h, h = π/N, at cell centres; exact cell weights w_j = cos x_{j−½} − cos x_{j+½}
    (Σ w = 2, so 2π Σ w = 4π is the round area); and the axisymmetric Laplacian Δ_{S²} in flux form,
    (Δu)_j = [sin x_{j+½}(u_{j+1} − u_j) − sin x_{j−½}(u_j − u_{j−1})] / (h w_j), as an (N, N) matrix.
    sin vanishes at the poles, so no flux leaves the sphere and Σ_j w_j (Δu)_j = 0 exactly."""
    h = math.pi / n
    edges = torch.arange(n + 1, dtype=torch.float64, device=device) * h  # x_{j−½}, j = 0 … N
    x = edges[:-1] + h / 2  # (N,)
    w = torch.cos(edges[:-1]) - torch.cos(edges[1:])  # (N,)
    s = torch.sin(edges)  # sin at the cell faces; s[0] = s[N] = 0 (up to round-off)
    s[0] = s[-1] = 0.0
    lap = torch.zeros(n, n, dtype=torch.float64, device=device)
    j = torch.arange(n, device=device)
    lap[j, j] = -(s[1:] + s[:-1]) / (h * w)
    lap[j[:-1], j[:-1] + 1] = s[1:-1] / (h * w[:-1])
    lap[j[1:], j[1:] - 1] = s[1:-1] / (h * w[1:])
    return x, w, lap


def lumpy_factors(x: torch.Tensor) -> torch.Tensor:
    """Three conformal factors u(x), metric e^{2u}(dx² + sin²x dθ²): a peanut, a pear and a lumpy sphere, built from
    Legendre polynomials P_l(cos x) (axisymmetric spherical harmonics).   (N,) → (3, N)"""
    c = torch.cos(x)
    p2 = (3 * c**2 - 1) / 2
    p3 = (5 * c**3 - 3 * c) / 2
    p4 = (35 * c**4 - 30 * c**2 + 3) / 8
    p6 = (231 * c**6 - 315 * c**4 + 105 * c**2 - 5) / 16
    return torch.stack([0.35 * p2 + 0.25 * p4, 0.12 * p3 + 0.2 * p2, 0.12 * p4 + 0.1 * p6 + 0.15 * p3])


def s3_grid(n: int, device: torch.device, a: float = 0.85) -> tuple:
    """Cells uniform in η ∈ (0, π), x = X(η) = η − (a/2) sin 2η: X′ = 1 − a cos 2η is 1 − a near the poles and
    1 + a at the equator, so the poles (where the bulbs of a dumbbell end up in conformal coordinates) get finer
    cells. X is odd about both poles, so mirror ghost cells still work. Returns (x, x_η, x_ηη, h), each (N,)."""
    h = math.pi / n
    eta = (torch.arange(n, dtype=torch.float64, device=device) + 0.5) * h
    return eta - a / 2 * torch.sin(2 * eta), 1 - a * torch.cos(2 * eta), 2 * a * torch.sin(2 * eta), h


def round_s3(r: float, x: torch.Tensor) -> torch.Tensor:
    """Round 3-sphere of radius r in the conformal form g = e^{2u}(dx² + sin²x g_{S²}): u = ln r.   → (1, N)"""
    return torch.full_like(x, math.log(r))[None]


def conformal_from_profile(s: torch.Tensor, psi: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """Turn a profile ψ(s), s ∈ [0, ℓ] with ψ = 0 only at the ends, into the conformal factor u on the grid x.
    Solving dx/ds = sin x / ψ gives ln tan(x/2) = ∫ ds/ψ + const; then u = ln(ψ / sin x). The constant (a conformal
    motion of the round S³) is fixed so the two ends sit symmetrically.   s, psi (M,) fine samples → (1, N)"""
    inner = slice(1, -1)  # the integral of 1/ψ diverges logarithmically at the poles
    ds = s[inner][1:] - s[inner][:-1]
    y = torch.cat(
        [torch.zeros(1, dtype=s.dtype), torch.cumsum(ds * 0.5 * (1 / psi[inner][1:] + 1 / psi[inner][:-1]), 0)]
    )
    y = y - 0.5 * (y[0] + y[-1] + torch.log((s[-1] - s[-2]) / s[1]))  # symmetric ends (ψ ≈ distance to the pole)
    xs = 2 * torch.atan(torch.exp(y))  # (M−2,) increasing from ≈ 0 to ≈ π
    us = torch.log(psi[inner] / torch.sin(xs))
    k = torch.searchsorted(xs, x).clamp(1, len(xs) - 1)  # linear interpolation onto the grid, all cells at once
    frac = ((x - xs[k - 1]) / (xs[k] - xs[k - 1])).clamp(0, 1)
    return (us[k - 1] + frac * (us[k] - us[k - 1]))[None]


def dumbbell(neck: float, x: torch.Tensor) -> torch.Tensor:
    """A symmetric dumbbell 3-sphere: profile ψ(s) = sin s · (1 − (1 − ε) e^{−((s − π/2)/0.35)²}), s ∈ [0, π]: two
    bulbs of radius ≈ 0.9 joined by a neck of radius ε, converted to the conformal factor u.   → (1, N)"""
    s = torch.linspace(0, math.pi, 20001, dtype=torch.float64, device=x.device)
    psi = torch.sin(s) * (1 - (1 - neck) * torch.exp(-(((s - math.pi / 2) / 0.35) ** 2)))
    return conformal_from_profile(s, psi, x)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/maths/poincare_ricci_flow/solve.py
"""Time stepping of both flows, and the surgery step that cuts a pinching neck and glues in a round cap."""

import math

import torch

from problems._common.progress import steps

from . import tensors
from .equation import area, gauss_curvature, neck_radius, profile, s3_rhs, surface_rhs


def flow_surface(u0: torch.Tensor, w: torch.Tensor, lap: torch.Tensor, t_max: float, n_save: int) -> dict:
    """Normalised Ricci flow on S² for a batch of conformal factors u0 (B, N), RK4 with the explicit stability limit
    dt ≈ 0.4 h² min e^{2u} / 2. Returns t (S,), u and K (S, B, N), area (S, B)."""
    n = u0.shape[-1]
    h = math.pi / n
    u = u0.clone()
    dt0 = 0.2 * h**2 * float(torch.exp(2 * u0).min())
    n_steps = max(1, math.ceil(t_max / dt0))
    dt = t_max / n_steps
    save_at = {round(k * n_steps / max(1, n_save - 1)) for k in range(n_save)}
    t, us = [], []
    for step in steps(n_steps + 1, "surface flow"):
        if step in save_at:
            t.append(step * dt)
            us.append(u)
        if step == n_steps:
            break
        k1 = surface_rhs(u, w, lap)
        k2 = surface_rhs(u + 0.5 * dt * k1, w, lap)
        k3 = surface_rhs(u + 0.5 * dt * k2, w, lap)
        k4 = surface_rhs(u + dt * k3, w, lap)
        u = u + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    u_all = torch.stack(us)  # (S, B, N)
    return {
        "t": torch.tensor(t, dtype=torch.float64),
        "u": u_all,
        "K": torch.stack([gauss_curvature(v, lap) for v in u_all]),
        "area": torch.stack([area(v, w) for v in u_all]),
    }


def flow_s3(
    u: torch.Tensor,
    grid: tuple,
    t_max: float,
    psi_stop: float,
    save_every: int = 20,
    radius_stop: float = 0.0,
) -> dict:
    """Ricci flow of g = e^{2u}(dx² + sin²x g_{S²}) for a batch u (B, N), forward Euler with dt = 0.2 min (h x_η e^u)²
    (the explicit diffusion limit). Stops at t_max, when the neck radius drops below psi_stop (time for surgery) or
    when max ψ < radius_stop (the piece is about to vanish). Returns per-step t and neck (T,) and snapshots of u
    (S, B, N) at t_save (S,)."""
    x, xe, _, h = grid
    u = u.clone()
    t, t_steps, ts, necks, us = 0.0, [], [], [], []
    step = 0
    while True:
        psi = torch.exp(u) * torch.sin(x)
        neck = neck_radius(psi).min().item()
        necks.append(neck)
        t_steps.append(t)
        done = t >= t_max - 1e-15 or neck < psi_stop or psi.max().item() < radius_stop or not math.isfinite(t)
        if step % save_every == 0 or done:
            ts.append(t)
            us.append(u.clone())
        if done:
            break
        dt = min(0.2 * h**2 * (xe**2 * torch.exp(2 * u)).min().item(), t_max - t)  # (physical cell size)²
        u = u + dt * s3_rhs(u, grid)
        t += dt
        step += 1
    return {
        "t": torch.tensor(t_steps, dtype=torch.float64),
        "neck": torch.tensor(necks, dtype=torch.float64),
        "t_save": torch.tensor(ts, dtype=torch.float64),
        "u": torch.stack(us),
    }


def surgery(u: torch.Tensor, grid: tuple, keep: float = 0.4) -> torch.Tensor:
    """A cartoon of Perelman's surgery: on the left piece, cut where ψ first falls to keep·max ψ (discarding the
    whole high-curvature neck region) and glue in a round spherical cap that meets the profile tangentially:
    ψ = R sin(θ₀ + (s − s_cut)/R) with R sin θ₀ = ψ_cut and cos θ₀ = ψ_s(cut), ending at θ = π. Then back to
    conformal form. (1, N) → (1, N). By symmetry the right piece is a mirror copy and evolves identically."""
    s, psi = profile(u, grid)  # (1, N) at the cell centres
    s, psi = s[0], psi[0]
    mid = psi[1:-1]
    is_min = (mid < psi[:-2]) & (mid < psi[2:])
    neck = int(torch.where(is_min, mid, torch.inf).argmin()) + 1  # cell of the neck
    top = int(psi[:neck].argmax())  # the left bulb
    j = top + int(torch.nonzero(psi[top:] < keep * psi[top])[0])  # first cell past the bulb below the cut level
    r = psi[j].item()
    slope = ((psi[j + 1] - psi[j - 1]) / (s[j + 1] - s[j - 1])).clamp(-0.99, 0.99).item()  # ψ_s at the cut
    theta0 = math.acos(slope)
    big_r = r / math.sin(theta0)
    zero = torch.zeros(1, dtype=s.dtype, device=s.device)
    s_cap = torch.linspace(0, big_r * (math.pi - theta0), 400, dtype=s.dtype, device=s.device)[1:]
    s_all = torch.cat([zero, s[: j + 1], s[j] + s_cap])
    psi_all = torch.cat([zero, psi[: j + 1], big_r * torch.sin(theta0 + s_cap / big_r)])
    psi_all[-1] = 0.0
    fine = torch.linspace(0, s_all[-1].item(), 20001, dtype=s.dtype, device=s.device)
    k = torch.searchsorted(s_all, fine).clamp(1, len(s_all) - 1)  # resample finely before converting
    frac = (fine - s_all[k - 1]) / (s_all[k] - s_all[k - 1])
    psi_fine = psi_all[k - 1] + frac * (psi_all[k] - psi_all[k - 1])
    return tensors.conformal_from_profile(fine, psi_fine, grid[0])


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/maths/poincare_ricci_flow/plot.py
"""Lumpy spheres rounding out (coloured by Gauss curvature), their curvature spread, and the dumbbell 3-sphere's
profile ψ(s) through neckpinch, surgery and rounding."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import LineCollection

from problems._common.plotting import ACCENT, save_gif

from .equation import embed_profile, profile


def _pick(n_have: int, n_want: int) -> list[int]:
    return sorted({round(k * (n_have - 1) / max(1, n_want - 1)) for k in range(n_want)})


def _neck(psi: np.ndarray) -> float:
    """Smallest interior local minimum of the profile (nan if there is none)."""
    mid = psi[1:-1]
    is_min = (mid < psi[:-2]) & (mid < psi[2:])
    return float(mid[is_min].min()) if is_min.any() else float("nan")


def animate(x, surf, spread, gb, grid, pre, post, n_frames: int, path: Path) -> Path:
    # 3-sphere frames: pre-surgery snapshots, then post-surgery ones
    n_post = 0 if post is None else n_frames * 2 // 5
    frames3 = [("pre", i) for i in _pick(len(pre["t_save"]), n_frames - n_post)]
    if post is not None:
        frames3 += [("post", i) for i in _pick(len(post["t_save"]), n_post)]
    n_total = len(frames3)
    surf_idx = [round((f / max(1, n_total - 1)) ** 2 * (len(surf["t"]) - 1)) for f in range(n_total)]  # slow start

    shapes = []
    for s in range(len(surf["t"])):
        rho, z, _ = embed_profile(surf["u"][s], x)
        shapes.append((rho.cpu().numpy(), z.cpu().numpy()))
    k_all = surf["K"].cpu().numpy()
    t_s, spr = surf["t"].cpu().numpy(), spread.cpu().numpy()

    def prof(u):
        s, psi = profile(u, grid)
        s, psi = s[0].cpu().numpy(), psi[0].cpu().numpy()
        length = s[-1] + (s[-1] - s[-2]) / 2
        return np.r_[0, s, length], np.r_[0, psi, 0]

    fig, (ax_a, ax_b, ax_c, ax_d) = plt.subplots(
        1, 4, figsize=(13, 3.6), gridspec_kw={"width_ratios": [1.5, 1, 1.4, 1]}
    )
    names = ["peanut", "pear", "lumpy"]
    offsets = [-2.6, 0.0, 2.6]
    norm = plt.Normalize(0, 2)
    lcs, fills = [], []
    for b, off in enumerate(offsets):
        lc = LineCollection([], cmap="coolwarm", norm=norm, lw=2.5)
        ax_a.add_collection(lc)
        lcs.append(lc)
        fills.append(None)
        ax_a.text(off, -1.75, names[b], ha="center", fontsize=8, color="#374151")
    ax_a.set_xlim(-4, 4)
    ax_a.set_ylim(-1.9, 1.6)
    ax_a.set_aspect("equal")
    ax_a.axis("off")
    fig.colorbar(lcs[0], ax=ax_a, fraction=0.03, pad=0.01).set_label("Gauss curvature $K$", fontsize=8)
    a_title = ax_a.set_title("", fontsize=9)

    for b in range(len(names)):
        ax_b.semilogy(t_s, spr[:, b], color=ACCENT[b], lw=1.5, label=names[b])
    cur_b = ax_b.axvline(0, color="#9ca3af", lw=1)
    ax_b.set_xlabel("$t$")
    ax_b.set_ylabel(r"$(K_\max - K_\min)/\bar K$", fontsize=8)
    ax_b.legend(fontsize=7)
    dev = np.abs(gb.cpu().numpy() / (4 * np.pi) - 1).max()
    ax_b.set_title(rf"$\int K\,dA = 4\pi\,(1 \pm {max(dev, 1e-16):.0e})$", fontsize=9)

    lengths = [prof(pre["u"][i])[0][-1] for i in range(len(pre["t_save"]))]
    if post is not None:
        lengths += [2 * prof(post["u"][i])[0][-1] for i in range(len(post["t_save"]))]
    ax_c.set_xlim(-0.05, max(lengths) * 1.03)
    ax_c.set_ylim(-1.0, 1.0)
    ax_c.set_aspect("equal")
    ax_c.set_xlabel("arclength $s$")
    ax_c.set_ylabel(r"$\pm\psi(s)$")
    c_title = ax_c.set_title("", fontsize=9)
    c_art = []

    t_pre, neck = pre["t"].cpu().numpy(), pre["neck"].cpu().numpy()
    ok = np.isfinite(neck)
    ax_d.plot(t_pre[ok], neck[ok] ** 2, color=ACCENT[0], lw=1.5, label=r"$\psi_\mathrm{neck}^2$")
    ax_d.set_ylim(0, None)
    ax_d.set_xlabel("$t$")
    ax_d.set_ylabel(r"neck radius$^2$", fontsize=8, color=ACCENT[0])
    if post is not None:
        ax_r = ax_d.twinx()
        t_post, ratio = post["t_save"].cpu().numpy(), post["ratio"].cpu().numpy()
        ax_r.semilogy(t_post, ratio - 1, color=ACCENT[1], lw=1.5)
        ax_r.set_ylabel(r"$K_\max/K_\min - 1$ (capped piece)", fontsize=8, color=ACCENT[1])
        ax_r.grid(False)
        ax_d.axvline(t_pre[-1], color="#9ca3af", ls="--", lw=1)
        ax_d.text(t_pre[-1], ax_d.get_ylim()[1] * 0.95, " surgery", fontsize=7, color="#6b7280", va="top")
    cur_d = ax_d.axvline(0, color="#9ca3af", lw=1)
    ax_d.set_title("neckpinch, then roundness", fontsize=9)

    def update(f: int):
        s = surf_idx[f]
        for b, off in enumerate(offsets):
            rho, z = shapes[s][0][b], shapes[s][1][b]
            xs, ys = np.r_[rho[::-1], -rho] + off, np.r_[z[::-1], z]
            kk = np.r_[k_all[s, b][::-1], k_all[s, b]]
            pts = np.c_[xs, ys].reshape(-1, 1, 2)
            lcs[b].set_segments(np.concatenate([pts[:-1], pts[1:]], axis=1))
            lcs[b].set_array(0.5 * (kk[:-1] + kk[1:]))
            if fills[b] is not None:
                fills[b].remove()
            fills[b] = ax_a.fill(xs, ys, color="#93c5fd", alpha=0.35, lw=0)[0]
        a_title.set_text(f"2-spheres, normalised flow, t = {t_s[s]:.2f}")
        cur_b.set_xdata([t_s[s]])

        while c_art:
            c_art.pop().remove()
        phase, i = frames3[f]
        if phase == "pre":
            sc, psi = prof(pre["u"][i])
            t_now = pre["t_save"][i].item()
            c_art.append(ax_c.fill_between(sc, -psi, psi, color="#93c5fd", alpha=0.6, lw=0))
            c_art.extend(ax_c.plot(sc, psi, color="#2563eb", lw=1.5) + ax_c.plot(sc, -psi, color="#2563eb", lw=1.5))
            c_title.set_text(rf"dumbbell $S^3$, $t = {t_now:.4f}$, neck $\psi = {_neck(psi):.3f}$")
        else:
            sc, psi = prof(post["u"][i])
            t_now = post["t_save"][i].item()
            for xs, alpha in ((sc, 0.6), (2 * sc[-1] - sc[::-1], 0.25)):  # the piece and its mirror image
                ps = psi if alpha > 0.5 else psi[::-1]
                c_art.append(ax_c.fill_between(xs, -ps, ps, color="#93c5fd", alpha=alpha, lw=0))
                c_art.extend(ax_c.plot(xs, ps, color="#2563eb", lw=1.5, alpha=alpha + 0.3))
                c_art.extend(ax_c.plot(xs, -ps, color="#2563eb", lw=1.5, alpha=alpha + 0.3))
            c_title.set_text(rf"after surgery: two capped pieces, $t = {t_now:.3f}$")
        cur_d.set_xdata([t_now])

    fig.tight_layout()
    return save_gif(fig, update, list(range(n_total)), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'neck': 0.15,
    't_surface': 3.0,
    'cells': 80
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.maths.poincare_ricci_flow")]:
    del sys.modules[name]
main = importlib.import_module("problems.maths.poincare_ricci_flow.main")
problem_dir = Path("problems/maths/poincare_ricci_flow")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
